# Isolation Forest: Anomalieerkennung (Praesentation)

Dieses Notebook basiert auf den Dateien `iso_forest_model.py`, `iso_forest_train.py`, `iso_forest_test.py` und `iso_forest_gridsearch.py`.

## 1. Setup und Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from itertools import product
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().parents[1]
SRC_DIR = PROJECT_ROOT / "01_src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_utils import read_data, sensor_cols
from iso_forest_model import fit_pipeline, predict_pipeline

## 2. Daten laden und vorbereiten

In [ ]:
train_path = PROJECT_ROOT / "02_data" / "train" / "train_FD001.txt"
test_path = PROJECT_ROOT / "02_data" / "test" / "test_FD001.txt"
rul_path = PROJECT_ROOT / "02_data" / "RUL" / "RUL_FD001.txt"

train_df = read_data(train_path)
test_df = read_data(test_path)
rul_df = pd.read_csv(rul_path, header=None, names=["RUL"])
rul_df["unit_id"] = rul_df.index + 1

threshold_std = 0.01
useful_sensors = [col for col in sensor_cols if train_df[col].std() > threshold_std]

max_cycles = train_df.groupby("unit_id")["cycles"].max().reset_index()
max_cycles.columns = ["unit_id", "max_cycle"]
train_df = train_df.merge(max_cycles, on="unit_id")
train_df["RUL"] = train_df["max_cycle"] - train_df["cycles"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(train_df[useful_sensors])
X_test_scaled = scaler.transform(test_df[useful_sensors])

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Anzahl Sensorsignale (nach Filter): {len(useful_sensors)}")

## 3. Trainingslauf (wie in iso_forest_train.py)

In [ ]:
n_estimators = 100
contamination = 0.03
max_samples = 128
window = 10
z_threshold = -2.5

train_run_df, model, score_mean, score_std = fit_pipeline(
    train_df, X_scaled, n_estimators, contamination, max_samples, window, z_threshold
)

y_true_train = (train_run_df["RUL"] <= 30).astype(int)
y_pred_train = train_run_df["anomaly_flag"].astype(int)

print(f"Precision: {precision_score(y_true_train, y_pred_train):.4f}")
print(f"Recall:    {recall_score(y_true_train, y_pred_train):.4f}")
print(f"F1:        {f1_score(y_true_train, y_pred_train):.4f}")

In [ ]:
first_anomaly = (
    train_run_df[train_run_df["anomaly_flag"] == 1]
    .groupby("unit_id")["RUL"]
    .max()
)
first_anomaly.describe()

## 4. Testlauf (wie in iso_forest_test.py)

In [ ]:
test_run_df = predict_pipeline(
    test_df, X_test_scaled, model, score_mean, score_std, window, z_threshold
)
test_run_df = test_run_df.merge(rul_df, on="unit_id", how="left")

units_with_alarm = test_run_df[test_run_df["anomaly_flag"] == 1]["unit_id"].nunique()
alarmed_units = test_run_df[test_run_df["anomaly_flag"] == 1]["unit_id"].unique()

print(f"Units mit Alarm: {units_with_alarm} / 100")
print("RUL der Units MIT Alarm:")
print(rul_df[rul_df["unit_id"].isin(alarmed_units)]["RUL"].describe())
print("RUL der Units OHNE Alarm:")
print(rul_df[~rul_df["unit_id"].isin(alarmed_units)]["RUL"].describe())

## 5. Gridsearch (wie in iso_forest_gridsearch.py)

In [ ]:
y_true_global = (train_df["RUL"] <= 30).astype(int)

param_grid = {
    "n_estimators": [100, 200],
    "contamination": [0.03, 0.05, 0.08],
    "max_samples": [128, 256],
    "window": [10, 15, 20],
    "z_threshold": [-1.5, -2.0, -2.5],
}

results = []
best_result = None

for n_estimators_, contamination_, max_samples_, window_, z_threshold_ in product(
    param_grid["n_estimators"],
    param_grid["contamination"],
    param_grid["max_samples"],
    param_grid["window"],
    param_grid["z_threshold"],
):
    candidate_df, _, _, _ = fit_pipeline(
        train_df, X_scaled, n_estimators_, contamination_, max_samples_, window_, z_threshold_
    )

    y_pred = candidate_df["anomaly_flag"].astype(int)
    precision = precision_score(y_true_global, y_pred, zero_division=0)
    recall = recall_score(y_true_global, y_pred, zero_division=0)
    f1 = f1_score(y_true_global, y_pred, zero_division=0)

    row = {
        "n_estimators": n_estimators_,
        "contamination": contamination_,
        "max_samples": max_samples_,
        "window": window_,
        "z_threshold": z_threshold_,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
    results.append(row)

    if best_result is None or (f1, recall, precision) > (best_result["f1"], best_result["recall"], best_result["precision"]):
        best_result = row

grid_df = pd.DataFrame(results).sort_values(by=["f1", "recall", "precision"], ascending=False)
grid_df.head(10)

In [ ]:
best_result

## 6. Folien-Kernaussagen

- Isolation Forest liefert pro Zeitpunkt einen Anomaly Score.
- Der Score wird pro Unit geglaettet (Rolling Mean) und z-normalisiert.
- Alarm-Logik: `score_normalized < z_threshold`.
- Gridsearch hilft bei der Balance zwischen frueher Erkennung und Fehlalarmen.